# 🧠 nnU-Net v2 on BraTS 2021 — brain tumor segmentation (research arm)

**Goal:** run nnU-Net on a BraTS 2021 subset with a **fixed, documented data split** that the
Swin UNETR arm (notebook 03) will reuse — this is what turns "running a framework" into a
**controlled comparison study**.

**📎 Required Kaggle Dataset:** [`awsaf49/brats-2021-task-data`](https://www.kaggle.com/datasets/awsaf49/brats-2021-task-data)
(Add Input → search "brats 2021 task data"). It mirrors the official RSNA-ASNR-MICCAI BraTS 2021
training data (1251 cases, 4 MRI modalities + segmentation).

**⚙️ Settings:** GPU T4 x2 (or better) · Internet On.

**⏱️ Budget (50 cases, 100 epochs, T4):** convert ~10 min · plan/preprocess ~30–60 min ·
train ~6–9 h · predict+eval ~20 min. **Tight but fits 12 h.** Scale knobs:
`MAX_CASES` (↓ faster) and `-tr nnUNetTrainer_250epochs` (↑ closer to SOTA, needs A100 or 2 sessions).

### BraTS label mapping (documented for reproducibility)
| Original value | Structure | Remapped value (this repo) |
|---|---|---|
| 0 | background | 0 |
| 1 | necrotic & non-enhancing tumor (NCR/NET) | 1 |
| 2 | peritumoral edema (ED) | 2 |
| 4 | gadolinium-enhancing tumor (ET) | **3** ← nnU-Net needs consecutive labels |

## 0 · Setup

In [ ]:
!nvidia-smi -L
!pip install -q -U nnunetv2

In [ ]:
import os, glob, json, pathlib
BASE = "/kaggle/working/nnunet_env"
for k in ("raw", "pre", "res"):
    os.makedirs(f"{BASE}/{k}", exist_ok=True)
os.environ["nnUNet_raw"]          = f"{BASE}/raw"
os.environ["nnUNet_preprocessed"] = f"{BASE}/pre"
os.environ["nnUNet_results"]      = f"{BASE}/res"

MAX_CASES   = 50     # <-- scale knob: 50 fits one T4 session; 100+ needs A100 / 2 sessions
EPOCHS      = 100    # nnUNetTrainer_{EPOCHS}epochs preset (built-in: 50/100/250/500)
DATASET_ID  = 701
CONFIG      = "3d_fullres"

# locate the attached BraTS dataset (name may vary slightly with the Kaggle mount)
cands = glob.glob("/kaggle/input/*brats*2021*") + glob.glob("/kaggle/input/*brats*")
assert cands, "Attach the Kaggle dataset 'awsaf49/brats-2021-task-data' first!"
BRATS_ROOT = cands[0]
train_dirs = glob.glob(f"{BRATS_ROOT}/**/RSNA_ASNR_MICCAI_BraTS_2021_TrainingData", recursive=True)
TRAIN_DIR = train_dirs[0] if train_dirs else BRATS_ROOT
cases = sorted(d for d in glob.glob(f"{TRAIN_DIR}/BraTS2021_*") if os.path.isdir(d))
print(f"BraTS root: {BRATS_ROOT}\n{len(cases)} total cases — using first {MAX_CASES}")

In [ ]:
!git clone --depth 1 https://github.com/naveed1011/nnunet-medseg-benchmark.git /kaggle/working/repo
REPO_DIR = "/kaggle/working/repo"
import sys; sys.path.insert(0, REPO_DIR)

## 1 · Fixed split FIRST (so nnU-Net and Swin UNETR see identical data)

We take the first `MAX_CASES` cases in sorted order, split **80/20 deterministic** (every 5th
case → validation), and inject this split into nnU-Net via its `splits_final.json` mechanism.
The same split file is exported to `/kaggle/working` so notebook 03 reuses it verbatim.
*Deterministic-by-design beats random here: anyone can reproduce it without a seed hunt.*

In [ ]:
case_ids_all = [os.path.basename(d) for d in cases[:MAX_CASES]]
train_ids = [c for i, c in enumerate(case_ids_all) if i % 5 != 4]
val_ids   = [c for i, c in enumerate(case_ids_all) if i % 5 == 4]
split = {"task": "BraTS2021", "max_cases": MAX_CASES, "note": "new ids are BraTS21_%05d by sorted original order",
         "id_map": {f"BraTS21_{i:05d}": c for i, c in enumerate(case_ids_all, 1)},
         "train": [f"BraTS21_{i:05d}" for i, c in enumerate(case_ids_all, 1) if (i-1) % 5 != 4],
         "val":   [f"BraTS21_{i:05d}" for i, c in enumerate(case_ids_all, 1) if (i-1) % 5 == 4]}
pathlib.Path("/kaggle/working/split").mkdir(exist_ok=True)
json.dump(split, open("/kaggle/working/split/brats_subset_split.json", "w"), indent=2)
print(f"train: {len(split['train'])}   val: {len(split['val'])}")
print("val cases:", split["val"])

## 2 · Convert to nnU-Net format (`Dataset701_BraTS21`)

The converter (from this repo) renames modalities to nnU-Net channel suffixes
(`flair,t1,t1ce,t2 → _0000.._0003`), **remaps label 4 → 3**, and writes a proper
`dataset.json`. Files are copied (not symlinked) so everything lives under `/kaggle/working`.

In [ ]:
%%time
!cd {REPO_DIR} && python data/convert_to_nnunet.py --task brats --raw {BRATS_ROOT} --max-cases {MAX_CASES} --copy --out $nnUNet_raw

In [ ]:
# NOTE: nnU-Net creates the preprocessed folder during plan_and_preprocess (next section),
# so we inject the fixed split AFTER planning and BEFORE training.
print("split file ready at /kaggle/working/split/brats_subset_split.json")

## 3 · Planning & preprocessing (3d_fullres only — Kaggle budget)

With 4 modalities at 1 mm isotropic BraTS resolution, preprocessing is the slowest step after
training. `-c 3d_fullres` skips the 2d config (brain tumors are inherently 3D structures, and
this also saves session time — a deliberate, documented choice).

In [ ]:
%%time
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity -c 3d_fullres -np 4

In [ ]:
# NOW inject the fixed split (nnU-Net created the preprocessed folder above).
# nnU-Net v2 reads `splits_final.json` (a list of {"train": [...], "val": [...]});
# fold 0 becomes our fixed split, so both model arms see identical cases.
pre_dir = os.path.join(os.environ["nnUNet_preprocessed"], f"Dataset{DATASET_ID}_BraTS21")
splits = [{"train": split["train"], "val": split["val"]}]
json.dump(splits, open(os.path.join(pre_dir, "splits_final.json"), "w"), indent=2)
print("injected fold_0 split:", len(splits[0]["train"]), "train /", len(splits[0]["val"]), "val")

In [ ]:
# 🔍 What did nnU-Net plan for BraTS?
plans = json.load(open(os.path.join(pre_dir, "nnUNetPlans.json")))
cfg = plans["configurations"]["3d_fullres"]
for key in ("patch_size", "batch_size", "spacing", "normalization_schemes", "resampling_fn_data"):
    print(f"{key:>24} : {cfg.get(key)}")

## 4 · Train fold 0 (the fixed split)

`--npz` stores validation softmax outputs. Checkpoints are written every 50 epochs
(`checkpoint_latest.*`) — the snapshot cell below protects them against Kaggle session loss.

In [ ]:
%%time
!nnUNetv2_train {DATASET_ID} {CONFIG} 0 -tr nnUNetTrainer_{EPOCHS}epochs --npz

### 💾 Snapshot results (run this BEFORE the session ends!)

`/kaggle/working` persists as notebook output; the env folders do not. Copy checkpoints +
progress + validation so you can (a) download them and (b) attach them to notebooks 04/05.
If a session dies mid-training: re-run setup+convert+preprocess, restore the snapshot into
`$nnUNet_results`, and continue with `nnUNetv2_train ... --c`.

In [ ]:
%%time
import shutil
res_dir = os.path.join(os.environ["nnUNet_results"], f"Dataset{DATASET_ID}_BraTS21")
snap = f"/kaggle/working/nnunet_snapshots/Dataset{DATASET_ID}_BraTS21"
if os.path.exists(snap): shutil.rmtree(snap)
shutil.copytree(res_dir, snap)
print("snapshot size: %.1f MB" % (sum(os.path.getsize(os.path.join(dp, fn))
      for dp, _, fs in os.walk(snap) for fn in fs) / 1024**2))
print("kept checkpoint_latest so a dead session can resume with --c")

## 5 · Predictions on the validation cases + evaluation

In [ ]:
%%time
# build an input folder with ONLY the validation cases (4 channels each)
val_in = "/kaggle/working/val_images_brats"; os.makedirs(val_in, exist_ok=True)
img_dir = os.path.join(os.environ["nnUNet_raw"], f"Dataset{DATASET_ID}_BraTS21", "imagesTr")
for cid in split["val"]:
    for f in glob.glob(os.path.join(img_dir, f"{cid}_*.nii.gz")):
        shutil.copy2(f, val_in)
print(len(os.listdir(val_in)), "files (", len(split["val"]), "cases x 4 modalities )")
!nnUNetv2_predict -i {val_in} -o /kaggle/working/predictions/brats_nnunet_val -d {DATASET_ID} -c {CONFIG} -f 0

In [ ]:
%%time
from src.metrics import evaluate_folders
gt_dir = os.path.join(os.environ["nnUNet_raw"], f"Dataset{DATASET_ID}_BraTS21", "labelsTr")
report = evaluate_folders("/kaggle/working/predictions/brats_nnunet_val", gt_dir, labels=(1, 2, 3))
for k, v in report.mean.items():
    print(f"{k:>10} ->", {kk: round(vv, 4) for kk, vv in v.items()})
report.save(pathlib.Path("/kaggle/working/results/brats_nnunet"))

In [ ]:
# package everything for the next notebooks (attach as Kaggle Dataset output)
import shutil
out = "/kaggle/working/outputs_brats"
os.makedirs(out, exist_ok=True)
for f in glob.glob("/kaggle/working/results/brats_nnunet/*"): shutil.copy2(f, out)
shutil.copy2("/kaggle/working/split/brats_subset_split.json", out)
for f in glob.glob(f"{snap}/**/progress.png", recursive=True): shutil.copy2(f, f"{out}/progress_nnunet.png")
for f in glob.glob(f"{snap}/**/validation/summary.json", recursive=True): shutil.copy2(f, f"{out}/nnunet_validation_summary.json")
for f in glob.glob(f"{snap}/**/plans.json", recursive=True): shutil.copy2(f, f"{out}/nnunet_plans.json")
print("📦 outputs:", os.listdir(out))

## ✅ Results checklist & honest notes

- [ ] `results/brats_nnunet/summary.json` — Dice/HD95 per label (NCR, ED, ET)
- [ ] `progress.png` — loss + pseudo-dice curves (copy into your repo's `docs/figures/`!)
- [ ] `brats_subset_split.json` — the shared split for notebook 03
- [ ] snapshot of `nnUNet_results` (for XAI in notebook 05)

**Write these caveats into your report** (professors respect this):
1. Subset size (MAX_CASES) & reduced epoch budget (EPOCHS) → numbers are *pipeline-validation
   quality*, not BraTS-challenge quality. State it plainly.
2. Predictions on val cases come from a model that never saw them — that's the honest number;
   ignore train-case predictions.
3. HD95 = ∞ when a structure is missed entirely (esp. small ET on some cases) — we report the
   % of inf cases rather than silently dropping them.